In [0]:
rubiks-project

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7609985695268105>, line 1
----> 1 rubiks-project

NameError: name 'rubiks' is not defined

In [0]:
df = spark.read.csv("/Volumes/workspace/default/rubiks_volume/rubiks_solves.csv",
                    header=True,
                    inferSchema=True)

display(df)

In [0]:
df_clean = df.filter(df.solve_time > 0)

df_clean = df_clean.dropDuplicates()

display(df_clean)

In [0]:
df_clean.selectExpr("avg(solve_time) as mean_solve_time").show()

In [0]:
df_clean.approxQuantile("solve_time",[0.5],0.01)

In [0]:
df_clean.selectExpr("stddev(solve_time)").show()

In [0]:
df_clean.stat.corr("moves","solve_time")

In [0]:
leaderboard = df_clean.groupBy("user_id") \
                      .avg("solve_time") \
                      .orderBy("avg(solve_time)")

display(leaderboard)

In [0]:
df_clean.write.mode("overwrite").saveAsTable("rubiks_clean_table")

In [0]:
%sql
SELECT user_id,
       AVG(solve_time) AS avg_time
FROM rubiks_clean_table
GROUP BY user_id
ORDER BY avg_time

In [0]:
trend = df_clean.groupBy("date") \
                .avg("solve_time") \
                .orderBy("date")

display(trend)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4799660635193594>, line 1
----> 1 trend = df_clean.groupBy("date") \
      2                 .avg("solve_time") \
      3                 .orderBy("date")
      5 display(trend)

NameError: name 'df_clean' is not defined

In [0]:
trend = spark.table("rubiks_clean_table") \
             .groupBy("date") \
             .avg("solve_time") \
             .orderBy("date")

display(trend)

date,avg(solve_time)
2024-01-02,18.5
2024-01-03,25.1
2024-01-04,15.3
2024-01-05,42.7
2024-01-06,12.9
2024-01-07,20.4
2024-01-08,30.2
2024-01-09,14.8
2024-01-10,39.5
2024-01-11,11.7


In [0]:
display(trend)

date,avg(solve_time)
2024-01-02,18.5
2024-01-03,25.1
2024-01-04,15.3
2024-01-05,42.7
2024-01-06,12.9
2024-01-07,20.4
2024-01-08,30.2
2024-01-09,14.8
2024-01-10,39.5
2024-01-11,11.7


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, rank

windowSpec = Window.orderBy("avg_time")

leaderboard_rank = spark.table("rubiks_clean_table") \
                        .groupBy("user_id") \
                        .avg("solve_time") \
                        .withColumnRenamed("avg(solve_time)", "avg_time") \
                        .withColumn("rank", rank().over(windowSpec))

display(leaderboard_rank)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


user_id,avg_time,rank
5,12.3,1
3,15.05,2
9,16.9,3
1,19.45,4
7,22.6,5
2,27.65,6
8,28.4,7
10,35.2,8
4,41.1,9
6,50.3,10
